In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath('../src'))
from embeddings import train_w2v_model, get_w2v_embeddings, get_sbert_embeddings

file_path = '../data/processed_v3.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig')
    
    print("Доступні колонки:", df.columns.tolist())
    
    df['lemma_text'] = df['lemma_text'].fillna('')
    
    print(f"Успішно завантажено {len(df)} рядків.")
else:
    print(f"Помилка: Файл за шляхом {file_path} не знайдено.")

Доступні колонки: ['text_v2', 'lemma_text', 'pos_seq']
Успішно завантажено 9522 рядків.


In [ ]:
print("Навчання Word2Vec моделі...")
w2v_model = train_w2v_model(df['lemma_text'].tolist())
w2v_vectors = get_w2v_embeddings(df['lemma_text'].tolist(), w2v_model)

print("Генерація SBERT ембедінгів (це може зайняти час)...")
sbert_vectors = get_sbert_embeddings(df['lemma_text'].tolist())

print(f"Word2Vec shape: {w2v_vectors.shape}")
print(f"SBERT shape: {sbert_vectors.shape}")

Навчання Word2Vec моделі...
Генерація SBERT ембедінгів (це може зайняти час)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4735.86it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 298/298 [01:27<00:00,  3.41it/s]


Word2Vec shape: (9522, 100)
SBERT shape: (9522, 384)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def extract_label(text):
    text = str(text).strip()
    if not text:
        return 0
    last_char = text[-1]
    return int(last_char) if last_char.isdigit() else 0

y = df['lemma_text'].apply(extract_label)

X_text = df['lemma_text'].apply(lambda x: str(x)[:-1] if str(x).strip()[-1].isdigit() else x)

print(f"Розподіл класів:\n{y.value_counts()}")

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(X_text)

def evaluate_model(X, y_labels, name):
    X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.2, random_state=42)
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}\n")
    return acc, f1

tfidf_acc, tfidf_f1 = evaluate_model(X_tfidf, y, "Baseline 1: TF-IDF")
w2v_acc, w2v_f1 = evaluate_model(w2v_vectors, y, "Baseline 3: Word2Vec")
sbert_acc, sbert_f1 = evaluate_model(sbert_vectors, y, "Baseline 4: SBERT")

Розподіл класів:
lemma_text
0    7377
1    2144
5       1
Name: count, dtype: int64
--- Baseline 1: TF-IDF ---
Accuracy: 0.8289
Macro F1: 0.7209

--- Baseline 3: Word2Vec ---
Accuracy: 0.7601
Macro F1: 0.4340

--- Baseline 4: SBERT ---
Accuracy: 0.9685
Macro F1: 0.9557



In [ ]:
def find_nearest_neighbors(query_text, vectors, df, top_k=5):
    query_vec = get_sbert_embeddings([query_text])[0].reshape(1, -1)
    
    similarities = cosine_similarity(query_vec, vectors).flatten()
    
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    return df.iloc[top_indices].copy(), similarities[top_indices]

query = "Ваш пошуковий запит тут"
results, scores = find_nearest_neighbors(query, sbert_vectors, df)
results['score'] = scores
print(results[['lemma_text', 'score']])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8668.44it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 17.49it/s]

                                             lemma_text     score
9131  завдяки ваш сервіс я могти легко знайти та зам...  0.580461
9091  ваш сайт надавати вичерпний інформація про пос...  0.573970
266                                       мій хіб це те  0.553545
8936      ваш система пошук враховувати різний параметр  0.552015
1719                  на сайт бути детальний опис товар  0.542432
